# P7 반환 후 원본 재현 검산 r2

본평가 24개를 다시 학습하지 않습니다. 기존 P5 cache와 P7 결과를 읽어 선택·예측·fidelity·CI를 재현합니다. 원래 feature/threshold/checkpoint를 바꾸지 않으며 불일치는 실패로 보존합니다.

**T4 GPU 런타임**에서 위부터 실행하세요. CPU probe refit와 1,000회 bootstrap 때문에 수 시간이 걸릴 수 있습니다. 결과는 Drive `boolean_interp_v1_4/P7_source_audit_r2`에 저장하며, 같은 노트북으로 개별 refit/완료 run부터 재개할 수 있습니다. 기존 P7 산출물은 읽기만 합니다. 검산 결과를 반환한 뒤 P7 완료 여부를 판정합니다.

## Drive에서 입력 번들 연결

기존 `v1_4_p7_source_audit_r2.zip`을 Google Drive의 **내 드라이브 → boolean_interp_v1_4** 폴더에 미리 올려 두세요. 번들을 새로 받을 필요는 없습니다.

첫 셀은 Drive를 마운트한 뒤 ZIP을 Colab 로컬 디스크로 복사하면서 SHA-256을 검사합니다. 브라우저 업로드 창은 사용하지 않습니다. 다른 폴더에 두었다면 `BUNDLE_IN_DRIVE`만 수정하세요. 복사 진행률을 표시하고, 같은 런타임에서 재실행하면 checksum이 맞는 로컬 ZIP을 재사용합니다.


In [ ]:
from google.colab import files, drive
from pathlib import Path
import hashlib, zipfile, subprocess, sys, os, time, json

drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/boolean_interp_v1_4')
# Drive에서 다른 폴더에 보관했다면 이 경로만 수정하세요.
BUNDLE_IN_DRIVE=BASE/'v1_4_p7_source_audit_r2.zip'
EXPECTED_SHA256='4750e1638805a1f522d5680e4b3cbededb7abb2ffb336bd354f2d9f07487fe39'
EXPECTED_BYTES=416208073
bundle=Path('/content/v1_4_p7_source_audit_r2.zip')

def file_sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(8*1024*1024),b''):
            digest.update(chunk)
    return digest.hexdigest()

cached=(bundle.is_file() and bundle.stat().st_size==EXPECTED_BYTES
        and file_sha256(bundle)==EXPECTED_SHA256)
if cached:
    print('checksum 확인 완료: 기존 Colab 로컬 번들을 재사용합니다.')
else:
    if not BUNDLE_IN_DRIVE.is_file():
        raise FileNotFoundError(f'Drive에 ZIP을 올린 뒤 다시 실행하세요: {BUNDLE_IN_DRIVE}')
    if BUNDLE_IN_DRIVE.stat().st_size!=EXPECTED_BYTES:
        raise ValueError('Drive ZIP 크기가 다릅니다. Drive 업로드가 완료됐는지 확인하세요.')
    partial=bundle.with_suffix('.zip.partial')
    digest=hashlib.sha256();copied=0;last_report=0
    with BUNDLE_IN_DRIVE.open('rb') as source,partial.open('wb') as target:
        for chunk in iter(lambda:source.read(8*1024*1024),b''):
            target.write(chunk);digest.update(chunk);copied+=len(chunk)
            if copied-last_report>=64*1024*1024 or copied==EXPECTED_BYTES:
                print(f'Drive → Colab: {copied/1e6:.1f} / {EXPECTED_BYTES/1e6:.1f} MB',flush=True)
                last_report=copied
    if copied!=EXPECTED_BYTES or digest.hexdigest()!=EXPECTED_SHA256:
        partial.unlink(missing_ok=True)
        raise ValueError('Drive ZIP checksum 불일치. 원본 번들을 다시 확인하세요.')
    partial.replace(bundle)
    print('복사 및 checksum 검증 완료')

ROOT=Path('/content/MI_P7_audit_r2');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    for name in z.namelist():
        assert (ROOT/name).resolve().is_relative_to(ROOT.resolve()), 'ZIP 경로 오류'
    z.extractall(ROOT)
SOURCE=BASE/'P5_r2';RETURNED=BASE/'P7_r1';OUT=BASE/'P7_source_audit_r2';OUT.mkdir(exist_ok=True)
assert (SOURCE/'contract.json').exists() and (RETURNED/'evaluation_complete.json').exists(), '기존 Drive 경로를 확인하세요'
print('준비 완료: 다음 셀부터 실행하세요.')


## 1. 기존 실행과 같은 수치 라이브러리 및 환경 기록
설치 후 재시작 안내가 나오면 런타임을 재시작하고 첫 셀부터 다시 실행합니다.

In [ ]:
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2'
log=subprocess.run([sys.executable,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pytest==8.4.2','matplotlib==3.10.8','torch==2.11.0','--extra-index-url','https://download.pytorch.org/whl/cu128'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUT/f'install_{time.time_ns()}.txt').write_text(log.stdout);print(log.stdout);log.check_returncode()
subprocess.run([sys.executable,'-m','pytest','tests_v1_4/test_p7.py','tests_v1_4/test_p7_source_audit.py','-q'],cwd=ROOT,check=True)


## 2. 원본 재현 감사

원본 반환 파일 56,919개의 checksum과 P5 cache hash를 먼저 검사합니다. 기존 P5 감사 기준선은 재사용하고, 새로운 probe는 train/validation에서 재계산해 저장된 선택과 비교합니다. Test는 저장된 계수와 선택으로만 평가합니다.

전체 test 예측/의미 CI, 80,000개 위치의 fidelity와 train dead 비율, 전체 causal validation bin을 확인합니다. GPU 패칭 재현 표본은 효과를 보지 않고 고정 corpus 순서의 suite별 첫 pair와 general test 첫 16개입니다. 전체 causal/replacement raw 수치는 로컬 반환 검사 대상으로, 전체 summary CI와 층간 비교는 여기서 다시 계산합니다.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'scripts/audit_v1_4_p7_sources.py'),'--root',str(ROOT),'--source',str(SOURCE),'--returned',str(RETURNED),'--output',str(OUT)],cwd=ROOT,check=True)


## 3. 완료·중단 결과 다운로드
위 셀이 실패해도 이 셀을 실행할 수 있습니다. 아래 ZIP과 checksum JSON을 이 작업에 전달하세요. 완료되지 않은 검산을 성공으로 표시하지 않습니다.

In [ ]:
archive=Path(f'/content/v1_4_p7_source_audit_return_{time.time_ns()}.zip')
subprocess.run([sys.executable,'-m','interp_v1_4.p7_runner','export','--output',str(OUT),'--archive',str(archive)],cwd=ROOT,check=True)
files.download(str(archive));files.download(str(archive.with_suffix('.sha256.json')))
